<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Deep-Learning/02-tensors-computation-graphs-pytorch.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Deep Learning guideline](Deep-Learning.html)

## **Tensors, Computation Graphs, and PyTorch** {#tensors-computation-graphs-pytorch}

A deep learning model may contain millions of parameters and hundreds of operations, but its execution is built from a small vocabulary: tensors hold numerical state, tensor operations transform that state, and a computation graph records how outputs depend on earlier values. Most implementation failures therefore begin with one of three questions:

- What does each tensor axis mean?
- Is the operation combining the intended elements?
- Is PyTorch recording the dependency needed for gradients, serialization, and device movement?

This chapter develops a discipline for answering those questions before a model becomes large. The objective is not to memorize every PyTorch method. It is to learn how to reason from **semantics -> shape -> operation -> graph -> system state**. Later chapters will add neural components, backpropagation theory, optimization, and scalable execution on top of this foundation.

### **Scalars, Vectors, Matrices, and Tensors** {#scalars-vectors-matrices-tensors}

A **tensor** is a rectangular, typed, multidimensional array. In deep learning software, the word includes familiar lower-rank objects:

| Object | Rank | Example shape | Typical interpretation |
|---|---:|---|---|
| Scalar | 0 | `[]` | one loss, probability, or learning rate |
| Vector | 1 | `[D]` | one feature vector or class-logit vector |
| Matrix | 2 | `[B, D]` | a batch of feature vectors or a weight matrix |
| Rank-3 tensor | 3 | `[B, L, D]` | batched sequences of token representations |
| Rank-4 tensor | 4 | `[B, C, H, W]` | batched channel-first images |

For a tensor $X \in \mathbb{R}^{d_1 \times d_2 \times \cdots \times d_k}$:

- $k$ is its **rank**, or number of axes;
- $(d_1,d_2,\ldots,d_k)$ is its **shape**;
- $d_j$ is the length of axis $j$;
- $\prod_{j=1}^{k} d_j$ is its number of elements.

Rank is not matrix rank. Tensor rank here counts axes; matrix rank measures linear independence. A tensor with shape `[2, 3]` always has tensor rank 2, while its matrix rank may be 1 or 2 depending on its values.

Values alone are not enough to describe a tensor. A practical tensor also has:

- a **dtype**, such as `float32`, `bfloat16`, or `int64`;
- a **device**, such as CPU, CUDA GPU, or Apple MPS;
- a **layout and stride**, which determine how logical indices map to storage;
- optional autograd metadata, including whether operations should be tracked.

This is why two tensors with equal numerical values may behave differently: one can be an integer label on the CPU, while another is a floating-point activation on a GPU with gradient tracking enabled.

<details>
<summary><strong>PyTorch: construct tensors and audit their metadata</strong></summary>

~~~python
import torch

scalar = torch.tensor(2.5)                         # shape []
vector = torch.tensor([1.0, -1.0, 0.5])           # shape [3]
matrix = torch.arange(12, dtype=torch.float32).reshape(3, 4)
image_batch = torch.zeros(8, 3, 32, 32)            # [B, C, H, W]
token_batch = torch.zeros(8, 128, 768)             # [B, L, D]


def describe(name: str, tensor: torch.Tensor) -> None:
    """Print the metadata needed for basic tensor debugging."""
    print(
        f"{name:12s}",
        f"shape={tuple(tensor.shape)}",
        f"rank={tensor.ndim}",
        f"elements={tensor.numel()}",
        f"dtype={tensor.dtype}",
        f"device={tensor.device}",
    )


for tensor_name, value in {
    "scalar": scalar,
    "vector": vector,
    "matrix": matrix,
    "images": image_batch,
    "tokens": token_batch,
}.items():
    describe(tensor_name, value)
~~~

</details>

The examples use two common symbolic conventions: $B$ for batch size, $C$ for channels, $H \times W$ for spatial dimensions, $L$ for sequence length, and $D$ for feature width. These letters are not properties stored in a tensor. PyTorch sees only axis lengths. The programmer must preserve their meaning.

**Application.** A batch of 8 RGB images and a batch of 8 token sequences can both be rank-4 or rank-3 after preprocessing, but their axes support different operations. Exchanging a channel axis with a spatial axis may still produce a valid tensor while silently changing the model's meaning.

**Comparison summary.** A Python number represents one value; a tensor represents values plus structural and execution metadata. Rank describes how many indices are required, shape describes the legal index range, and axis semantics describe what those indices mean in the problem.

### **Tensor Shape, Axes, and Layout** {#tensor-shape-axes-layout}

A tensor's **shape** is a tuple of axis lengths. An **axis** is one coordinate direction, while an axis's **semantic role** explains what varying that coordinate changes. Consider an image batch $X \in \mathbb{R}^{B \times C \times H \times W}$:

$$
X[b,c,h,w]
$$

selects one scalar at batch item $b$, channel $c$, row $h$, and column $w$. The shape `[32, 3, 224, 224]` is useful only when it is paired with the interpretation `[batch, channel, height, width]`.

![A rank-4 tensor can assign a distinct meaning to each axis, such as batch, width, height, and feature.](assets/tf-tensor-axis-order.png){fig-align="center" width="76%" fig-alt="A rank-4 tensor diagram labeling the batch, width, height, and feature axes."}

*Image source: TensorFlow Core, [Introduction to Tensors](https://www.tensorflow.org/guide/tensor), licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). The TensorFlow diagram illustrates a framework-independent tensor concept; the code in this chapter uses PyTorch.*

Different libraries and kernels may choose different conventions. Vision models commonly use channel-first `[B, C, H, W]` in PyTorch, while some systems use channel-last `[B, H, W, C]`. Sequence models often use `[B, L, D]`, but older recurrent interfaces may use `[L, B, D]`. A permutation can convert between conventions:

$$
[B,H,W,C] \xrightarrow{\operatorname{permute}(0,3,1,2)} [B,C,H,W].
$$

Logical shape does not fully describe physical storage. A dense tensor has an underlying one-dimensional storage and a **stride** for each axis. If the stride tuple is $(s_1,\ldots,s_k)$, the logical element at index $(i_1,\ldots,i_k)$ is found at an offset proportional to:

$$
\operatorname{offset}
= i_1s_1+i_2s_2+\cdots+i_ks_k.
$$

For a row-major contiguous tensor with shape `[2, 3, 4]`, the stride is typically `[12, 4, 1]`: moving one step along the final axis advances one storage element, moving along the middle axis advances four, and moving along the first axis advances twelve.

Operations such as `transpose` and `permute` often create a **view** with new shape and stride metadata rather than copying values. The result can be non-contiguous. This is efficient, but a later operation that assumes contiguous storage may require `.contiguous()` or may make an implicit copy through `.reshape()`.

<details>
<summary><strong>PyTorch: inspect strides, views, permutations, and copies</strong></summary>

~~~python
import torch

x = torch.arange(24).reshape(2, 3, 4)
y = x.permute(0, 2, 1)  # [2, 4, 3], metadata changes; storage is shared.

print("x:", x.shape, x.stride(), x.is_contiguous())
print("y:", y.shape, y.stride(), y.is_contiguous())
print("shared storage:", x.untyped_storage().data_ptr() == y.untyped_storage().data_ptr())

# view() requires a compatible stride pattern. Flattening this permutation fails.
try:
    y.view(-1)
except RuntimeError as error:
    print("view failed:", str(error).split(".")[0])

# reshape() returns a view when possible and otherwise creates a contiguous copy.
y_flat = y.reshape(-1)
print("reshape is contiguous:", y_flat.is_contiguous())
print(
    "reshape shares storage:",
    y_flat.untyped_storage().data_ptr() == y.untyped_storage().data_ptr(),
)

# Make the copy explicit when a downstream kernel requires contiguous storage.
y_contiguous = y.contiguous()
assert y_contiguous.shape == (2, 4, 3)
assert y_contiguous.is_contiguous()
~~~

</details>

The difference matters for both correctness and performance. `permute` changes which axis an index refers to; `reshape` changes how existing elements are grouped; neither operation should be used to imitate the other. A tensor with the expected total number of elements can still be semantically wrong after an accidental reshape.

A useful practice is to maintain **shape invariants** at module boundaries:

~~~text
input images:       [B, C, H, W]
patch embeddings:   [B, L, D]
class logits:       [B, K]
targets:            [B]
~~~

Assertions such as `assert logits.shape == (targets.shape[0], number_of_classes)` turn silent semantic errors into immediate failures.

**Comparison summary.** Shape describes the logical grid, stride describes navigation through storage, and axis names describe meaning. `permute` reorders axes, `reshape` regroups elements, and `.contiguous()` materializes a compatible memory order when necessary.

### **Indexing, Reshaping, and Broadcasting** {#indexing-reshaping-broadcasting}

Indexing answers **which elements remain**; reshaping answers **how the same elements are organized**; broadcasting answers **how compatible shapes participate in an elementwise operation**. They are related but not interchangeable.

With a tensor `x` of shape `[B, L, D]`:

| Expression | Output shape | Meaning |
|---|---|---|
| `x[0]` | `[L, D]` | remove the batch axis and select one item |
| `x[0:1]` | `[1, L, D]` | keep a length-one batch axis |
| `x[:, -1]` | `[B, D]` | select the last position for every item |
| `x[..., 0]` | `[B, L]` | select the first feature along the final axis |
| `x.unsqueeze(1)` | `[B, 1, L, D]` | insert a length-one axis |
| `x.flatten(0, 1)` | `[B L, D]` | merge adjacent batch and sequence axes |

Basic slicing usually creates a view. Boolean or integer-array advanced indexing may allocate a new tensor. In-place modification of a view can also modify its base tensor, which is useful when intentional and dangerous when hidden.

Reshaping must preserve the number of elements:

$$
\prod_j d_j = \prod_m d'_m.
$$

For example, `[B, L, H, d_h]` can become `[B, L, H d_h]`, but that transformation is meaningful only if the final two axes really are attention heads and per-head features stored in the expected order.

Broadcasting aligns shapes from the **trailing axis**. Two aligned dimensions are compatible when they are equal, one of them is 1, or one dimension is absent. If $X$ has shape `[B, L, D]` and $b$ has shape `[D]`, then:

$$
X + b:
[B,L,D] + [D]
\longrightarrow [B,L,D],
$$

because `[D]` behaves as `[1, 1, D]`. The expanded values are normally represented by stride metadata rather than physically copied.

![Broadcasting combines a column tensor of shape 3 by 1 with a row tensor of shape 1 by 4 to produce a 3 by 4 result.](assets/tf-broadcasting.png){fig-align="center" width="68%" fig-alt="A broadcasting diagram multiplying a three-element column by a four-element row to form a three by four matrix."}

*Image source: TensorFlow Core, [Introduction to Tensors: Broadcasting](https://www.tensorflow.org/guide/tensor#broadcasting), licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). PyTorch follows the same NumPy-style trailing-axis rules; see the official [PyTorch broadcasting semantics](https://docs.pytorch.org/docs/stable/notes/broadcasting.html).*

Broadcasting is powerful because it removes explicit loops, but it can also produce a legal and unintended output. A classic example is adding `[B, 1]` to `[B]`: trailing-axis alignment produces `[B, B]`, not `[B, 1]`.

<details>
<summary><strong>PyTorch: normalize by channel and expose a silent broadcasting bug</strong></summary>

~~~python
import torch

# Four RGB images: [batch, channel, height, width].
images = torch.randn(4, 3, 16, 16)

# keepdim=True preserves [1, C, 1, 1], so every semantic axis remains visible.
channel_mean = images.mean(dim=(0, 2, 3), keepdim=True)
channel_std = images.std(dim=(0, 2, 3), keepdim=True).clamp_min(1e-6)
normalized = (images - channel_mean) / channel_std

assert channel_mean.shape == (1, 3, 1, 1)
assert normalized.shape == images.shape

# A legal but incorrect broadcast: [B, 1] + [B] -> [B, B].
scores = torch.randn(4, 1)
offsets = torch.arange(4, dtype=torch.float32)
wrong = scores + offsets
print("wrong shape:", wrong.shape)  # torch.Size([4, 4])

# Insert the intended singleton feature axis: [B] -> [B, 1].
correct = scores + offsets.unsqueeze(1)
assert correct.shape == scores.shape

# torch.broadcast_shapes can validate a planned elementwise operation explicitly.
assert torch.broadcast_shapes(images.shape, channel_mean.shape) == images.shape
~~~

</details>

For reductions such as `sum`, `mean`, or `max`, deciding whether to keep a reduced axis is equally important. `keepdim=True` often makes subsequent broadcasting safer because the semantic position of the axis remains explicit.

**Application.** Broadcasting supplies per-channel image normalization, per-feature affine transformations, attention masks, and bias addition. Each use should be explainable axis by axis. If an output unexpectedly gains a dimension, print both operand shapes before inspecting values.

**Comparison summary.** Indexing selects data, reshaping reorganizes the index system, and broadcasting virtually expands singleton dimensions for elementwise operations. Valid shapes guarantee that an operation can execute; they do not guarantee that it expresses the intended semantics.

### **Matrix Multiplication and Einstein Summation** {#matrix-multiplication-einstein-summation}

Elementwise multiplication and matrix multiplication answer different questions. For matrices $A,B \in \mathbb{R}^{M \times N}$, the Hadamard product preserves shape:

$$
(A \odot B)_{ij}=A_{ij}B_{ij}.
$$

Matrix multiplication contracts one axis. If $A \in \mathbb{R}^{M \times K}$ and $B \in \mathbb{R}^{K \times N}$, then:

$$
C=AB \in \mathbb{R}^{M \times N},
\qquad
C_{ij}=\sum_{k=1}^{K}A_{ik}B_{kj}.
$$

The shared $K$ axis disappears because its contributions are summed. This **contracted axis** is the central idea behind linear layers, dot-product attention, convolution implementations, and many losses.

PyTorch's `torch.matmul` and `@` generalize matrix multiplication:

- vector `@` vector returns a scalar dot product;
- matrix `@` matrix performs ordinary matrix multiplication;
- higher-rank inputs treat the final two axes as matrices and broadcast earlier batch axes.

For a sequence projection:

$$
X \in \mathbb{R}^{B \times L \times D_{in}},
\quad
W \in \mathbb{R}^{D_{in} \times D_{out}},
\quad
XW \in \mathbb{R}^{B \times L \times D_{out}}.
$$

Every token vector is multiplied by the same weight matrix. The leading `[B, L]` axes are preserved, while $D_{in}$ is contracted and replaced by $D_{out}$.

**Einstein summation** makes this axis logic explicit. In an equation such as:

$$
Y_{bld}=\sum_k X_{blk}W_{kd},
$$

an index appearing in the inputs but not the output is summed; an index appearing in the output is preserved. PyTorch expresses the same operation as `torch.einsum("blk,kd->bld", x, weight)`.

<details>
<summary><strong>PyTorch: compare elementwise, matrix, batched, and einsum operations</strong></summary>

~~~python
import math
import torch

B, L, D_IN, D_OUT = 2, 5, 4, 6
x = torch.randn(B, L, D_IN)
weight = torch.randn(D_IN, D_OUT)

# The final input-feature axis is contracted with weight axis 0.
projected_matmul = x @ weight
projected_einsum = torch.einsum("blk,kd->bld", x, weight)

assert projected_matmul.shape == (B, L, D_OUT)
assert torch.allclose(projected_matmul, projected_einsum)

# Attention scores contract the per-head feature axis d.
HEADS, HEAD_DIM = 3, 8
queries = torch.randn(B, HEADS, L, HEAD_DIM)
keys = torch.randn(B, HEADS, L, HEAD_DIM)

scores = torch.einsum("bhid,bhjd->bhij", queries, keys) / math.sqrt(HEAD_DIM)
assert scores.shape == (B, HEADS, L, L)

# Elementwise multiplication does not contract an axis.
gated_queries = queries * torch.sigmoid(keys)
assert gated_queries.shape == queries.shape
~~~

</details>

`einsum` is most useful when it clarifies a nontrivial contraction, not when it merely shortens familiar code. Its labels should follow a documented convention, and repeated dimensions should be checked with assertions. A syntactically valid equation can still contract the wrong axis.

Complexity follows the contracted dimensions. Multiplying `[M, K] @ [K, N]` requires on the order of $MKN$ multiply-add pairs, while the output stores $MN$ values. Batched leading axes multiply the work but do not change the contraction rule.

**Application.** A linear classifier contracts feature width into class logits; attention contracts query and key features into pairwise position scores; a bilinear model may contract two feature axes around a learned relation matrix. Writing the index equation before the code often reveals shape errors immediately.

**Comparison summary.** Elementwise operations align and preserve axes; matrix multiplication contracts one pair of axes; `einsum` describes arbitrary preservation and contraction with named index symbols. Shape reasoning is therefore a compact form of algorithm reasoning.

### **Computation Graphs** {#computation-graphs}

A **computation graph** represents a numerical program as dependencies. Tensor values flow along edges, and operation nodes transform incoming values into outputs. If a scalar objective is:

$$
q=3a^3-b^2,
$$

then the forward computation contains power, multiplication, and subtraction operations. The graph records enough local information to answer how changing $a$ or $b$ would change $q$.

A feed-forward execution for one model call normally forms a **directed acyclic graph (DAG)**: edges point from earlier values to later values, and no output depends on itself through a forward-time cycle. A recurrent neural network still produces a DAG after its repeated state transition is unfolded across a finite sequence.

![PyTorch records backward-function nodes for the operations that created an output tensor.](assets/pytorch-autograd-dag.png){fig-align="center" width="48%" fig-alt="A PyTorch autograd graph containing power, multiplication, and subtraction backward nodes."}

*Image source: PyTorch Tutorials, [A Gentle Introduction to `torch.autograd`](https://docs.pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html#computational-graph).*

In PyTorch, eager operations execute immediately while autograd dynamically records differentiable dependencies. During a forward pass it:

1. computes each requested tensor result;
2. attaches a `grad_fn` to non-leaf outputs when gradient tracking is required;
3. saves selected values or metadata needed by backward formulas.

The graph is **dynamic**: Python control flow can choose different operations on different calls. After a normal backward pass, saved graph state is released, and the next forward pass builds a new graph. This differs from a static source-code diagram: the autograd graph describes operations that actually ran for particular tensor values.

PyTorch distinguishes:

- **leaf tensors**, which are not produced by a tracked operation and often include trainable parameters;
- **non-leaf tensors**, which are intermediate results with a `grad_fn`;
- **roots**, typically scalar losses from which reverse traversal begins.

<details>
<summary><strong>PyTorch: inspect leaf tensors and the dynamic graph</strong></summary>

~~~python
import torch

a = torch.tensor([2.0, 3.0], requires_grad=True)
b = torch.tensor([6.0, 4.0], requires_grad=True)

q = 3 * a**3 - b**2
loss = q.sum()

print("a is leaf:", a.is_leaf, "grad_fn:", a.grad_fn)
print("q is leaf:", q.is_leaf, "grad_fn:", type(q.grad_fn).__name__)
print("loss grad_fn:", type(loss.grad_fn).__name__)

loss.backward()

# dq/da = 9a^2 and dq/db = -2b.
assert torch.allclose(a.grad, 9 * a.detach() ** 2)
assert torch.allclose(b.grad, -2 * b.detach())
print("gradient for a:", a.grad)
print("gradient for b:", b.grad)
~~~

</details>

The graph is not the same as the `nn.Module` tree. A module tree describes registered components and parameters; the computation graph describes one execution and can invoke the same module multiple times, skip a branch, or reuse a tensor along several paths.

Graph recording also explains why some in-place operations are unsafe. Backward may need an earlier tensor value. If that value is overwritten, PyTorch either raises a versioning error or, in poorly isolated custom code, the intended derivative becomes impossible to reconstruct.

**Comparison summary.** Source code describes possible execution, a module tree describes owned components, and a computation graph describes actual numerical dependencies. The graph connects forward values to the local derivative rules needed later.

### **Automatic Differentiation in PyTorch** {#automatic-differentiation-pytorch}

**Automatic differentiation (AD)** computes derivatives by applying exact derivative rules to the executed primitive operations. It is not symbolic differentiation, which manipulates algebraic expressions, and it is not finite differencing, which estimates slopes by perturbing inputs.

Deep learning usually has many parameters and one scalar loss. **Reverse-mode AD** is efficient for this many-input, one-output structure. If intermediate states satisfy:

$$
\mathbf{h}^{(0)}=\mathbf{x},
\qquad
\mathbf{h}^{(\ell)}=f^{(\ell)}(\mathbf{h}^{(\ell-1)}),
\qquad
\mathcal{L}=g(\mathbf{h}^{(L)}),
$$

reverse mode propagates an upstream sensitivity backward using vector-Jacobian products:

$$
\mathbf{v}_{\ell-1}
=
\left(\frac{\partial \mathbf{h}^{(\ell)}}
{\partial \mathbf{h}^{(\ell-1)}}\right)^{\!T}
\mathbf{v}_{\ell}.
$$

It does not normally materialize the full Jacobian. Each operation receives an upstream vector and returns the contribution required by its inputs. Chapter 04 develops this mechanism and backpropagation in detail; this section focuses on the PyTorch contract.

Setting `requires_grad=True` on a leaf tells autograd to track operations that can influence it. Calling `.backward()` on a scalar output starts reverse traversal. Gradients accumulate into the `.grad` fields of qualifying leaf tensors, so training code must clear them deliberately with `optimizer.zero_grad()` or set them to `None`.

Important interfaces have distinct purposes:

| Interface | Effect | Typical use |
|---|---|---|
| `tensor.detach()` | returns a tensor disconnected from the current graph | stop a gradient path or log a value |
| `torch.no_grad()` | disables graph recording inside a block | parameter updates or evaluation code |
| `torch.inference_mode()` | stronger inference-only optimization | deployed/evaluation forward passes |
| `torch.autograd.grad()` | returns selected gradients without relying only on `.grad` | penalties, analysis, higher-order methods |
| `retain_graph=True` | keeps graph state after backward | exceptional repeated traversals, with extra memory cost |

Broadcasting affects backward rules. If a bias $b \in \mathbb{R}^{D}$ is broadcast over a batch $X \in \mathbb{R}^{B \times D}$, the gradient for $b$ sums contributions over the expanded batch axis:

$$
\frac{\partial \mathcal{L}}{\partial b_d}
=
\sum_{i=1}^{B}
\frac{\partial \mathcal{L}}{\partial Y_{id}}.
$$

<details>
<summary><strong>PyTorch: verify an autograd gradient with finite differences</strong></summary>

~~~python
import torch

# float64 makes the numerical finite-difference comparison more precise.
x = torch.tensor([1.5, -2.0], dtype=torch.float64)
target = torch.tensor(0.4, dtype=torch.float64)
w = torch.tensor([0.3, -0.7], dtype=torch.float64, requires_grad=True)


def objective(weights: torch.Tensor) -> torch.Tensor:
    prediction = (x * weights).sum()
    return (prediction - target) ** 2


loss = objective(w)
loss.backward()
autograd_gradient = w.grad.detach().clone()

# Central finite differences are a diagnostic, not the training algorithm.
epsilon = 1e-6
finite_difference = torch.empty_like(w)
with torch.no_grad():
    for index in range(w.numel()):
        plus = w.detach().clone()
        minus = w.detach().clone()
        plus[index] += epsilon
        minus[index] -= epsilon
        finite_difference[index] = (
            objective(plus) - objective(minus)
        ) / (2 * epsilon)

print("autograd:", autograd_gradient)
print("finite difference:", finite_difference)
print("maximum error:", (autograd_gradient - finite_difference).abs().max().item())
assert torch.allclose(autograd_gradient, finite_difference, atol=1e-8, rtol=1e-6)
~~~

</details>

Finite differences are useful for checking a small custom operation, but they scale poorly and suffer from truncation and floating-point errors. AD reuses the actual program and computes derivatives to working precision, subject to the differentiability and numerical behavior of its primitives.

Common autograd failures include detaching a value too early, converting a tensor to a Python number with `.item()` before it contributes to the loss, modifying saved tensors in place, forgetting that gradients accumulate, or expecting a non-leaf tensor's `.grad` to be retained automatically.

**Comparison summary.** Finite differences estimate derivatives through repeated evaluations; symbolic differentiation transforms formulas; autograd composes local derivative rules along the executed graph. PyTorch's default reverse mode is well matched to scalar-loss training.

### **Parameters, Modules, and Model Composition** {#parameters-modules-model-composition}

An `nn.Module` is more than a Python function. It is an ownership boundary that registers trainable parameters, persistent buffers, and nested modules. Registration lets PyTorch discover system state for optimization, device movement, serialization, and mode changes.

An `nn.Parameter` is a tensor that becomes part of a module's parameter set when assigned as an attribute. A **buffer** is persistent tensor state that should move and serialize with the module but is not optimized by default. Examples include normalization statistics, masks, or fixed positional values.

Calling `model(inputs)` invokes the module's `__call__` machinery and then `forward`; calling `model.forward(inputs)` directly bypasses hooks and should normally be avoided. Nested modules form a tree, while a forward call through that tree creates the dynamic computation graph discussed above.

The `state_dict` maps registered state names to tensors. It normally contains parameters and persistent buffers, but not arbitrary Python attributes. This makes registration behavior observable:

<details>
<summary><strong>PyTorch: register parameters, buffers, and nested modules</strong></summary>

~~~python
import torch
from torch import nn


class StandardizedRegressor(nn.Module):
    def __init__(self, feature_mean: torch.Tensor, feature_std: torch.Tensor):
        super().__init__()

        # Buffers move with .to(device) and appear in state_dict,
        # but an optimizer will not update them.
        self.register_buffer("feature_mean", feature_mean)
        self.register_buffer("feature_std", feature_std.clamp_min(1e-6))

        # Assigning nn.Modules registers their Parameters recursively.
        self.predictor = nn.Sequential(
            nn.Linear(feature_mean.numel(), 8),
            nn.ReLU(),
            nn.Linear(8, 1),
        )

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        assert features.shape[-1] == self.feature_mean.numel()
        standardized = (features - self.feature_mean) / self.feature_std
        return self.predictor(standardized).squeeze(-1)


model = StandardizedRegressor(
    feature_mean=torch.tensor([10.0, 2.0, -1.0]),
    feature_std=torch.tensor([2.0, 0.5, 4.0]),
)
batch = torch.tensor([[12.0, 2.5, 3.0], [8.0, 1.5, -5.0]])
predictions = model(batch)

print("output shape:", predictions.shape)
print("parameter names:", [name for name, _ in model.named_parameters()])
print("buffer names:", [name for name, _ in model.named_buffers()])
print("state keys:", list(model.state_dict()))

assert predictions.shape == (2,)
assert "feature_mean" in model.state_dict()
assert "predictor.0.weight" in model.state_dict()
~~~

</details>

`model.train()` and `model.eval()` change the behavior of modules such as dropout and batch normalization; they do not enable or disable gradient recording. Evaluation commonly needs both `model.eval()` and `torch.inference_mode()` because they control different mechanisms.

Composition should preserve clear contracts. A module should document expected shape, dtype, value range, and output meaning. Small modules are easier to test, but excessive fragmentation can hide the main data flow. The best boundary usually owns a coherent transformation and its state.

**Comparison summary.** A tensor stores numerical state, a `Parameter` marks optimizable state, a buffer marks persistent non-optimized tensor state, and a `Module` owns and composes that state. The module tree is persistent structure; the computation graph is per-execution behavior.

### **Datasets, DataLoaders, and Mini-Batches** {#datasets-dataloaders-mini-batches}

A model consumes tensors, but real data begins as files, records, sequences, labels, and transformations. PyTorch separates this input system into responsibilities:

- a **Dataset** defines how a sample is identified and retrieved;
- a **Sampler** defines which sample indices are requested and in what order;
- a **collate function** combines retrieved samples into a batch;
- a **DataLoader** coordinates batching, iteration, workers, and memory-transfer options.

A map-style `Dataset` normally implements `__len__` and `__getitem__`. An `IterableDataset` yields a stream and is more suitable when random indexing is impossible or undesirable, such as logs, remote streams, or generated samples. The choice changes how shuffling and multi-worker partitioning must be handled.

Mini-batches serve both statistical and systems purposes. For per-example losses $\ell_i(\theta)$, a batch of size $B$ estimates the full-data gradient with:

$$
\widehat{\mathbf{g}}_B
=
\frac{1}{B}\sum_{i \in \mathcal{B}}
\nabla_{\theta}\ell_i(\theta).
$$

A larger batch generally reduces sampling noise and improves hardware utilization until memory or communication becomes limiting. It does not automatically improve generalization, and it changes the number of optimizer updates per epoch. Batch size, learning rate, normalization behavior, and scheduler units therefore need to be interpreted together.

The collate stage defines the batch contract. Fixed-size image samples may stack directly into `[B, C, H, W]`. Variable-length sequences cannot be stacked without a policy such as padding, packing, truncation, or nested/ragged representation. Padding also requires a mask so later operations can distinguish real positions from placeholders.

<details>
<summary><strong>PyTorch: build a variable-length Dataset and a mask-aware collate function</strong></summary>

~~~python
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset


class SequenceDataset(Dataset):
    def __init__(self):
        self.examples = [
            (torch.tensor([4, 8, 2, 9]), 1),
            (torch.tensor([3, 5]), 0),
            (torch.tensor([7, 1, 6]), 1),
            (torch.tensor([2, 2, 4, 8, 5]), 0),
        ]

    def __len__(self) -> int:
        return len(self.examples)

    def __getitem__(self, index: int):
        tokens, label = self.examples[index]
        return tokens, torch.tensor(label, dtype=torch.long)


def pad_collate(batch):
    """Convert a list of variable-length examples into dense tensors."""
    token_sequences, labels = zip(*batch)
    lengths = torch.tensor([sequence.numel() for sequence in token_sequences])
    padded = pad_sequence(token_sequences, batch_first=True, padding_value=0)

    # True marks a real token; False marks padding.
    positions = torch.arange(padded.shape[1]).unsqueeze(0)
    attention_mask = positions < lengths.unsqueeze(1)
    return {
        "tokens": padded,                  # [B, L_max]
        "attention_mask": attention_mask,  # [B, L_max]
        "lengths": lengths,                # [B]
        "labels": torch.stack(labels),     # [B]
    }


generator = torch.Generator().manual_seed(7)
loader = DataLoader(
    SequenceDataset(),
    batch_size=3,
    shuffle=True,
    collate_fn=pad_collate,
    num_workers=0,
    generator=generator,
)

batch = next(iter(loader))
for name, value in batch.items():
    print(name, tuple(value.shape), value.dtype)

assert batch["tokens"].shape == batch["attention_mask"].shape
assert torch.equal(batch["attention_mask"].sum(dim=1), batch["lengths"])
~~~

</details>

Shuffling should normally be enabled for independent training examples and disabled for deterministic evaluation. `drop_last=True` discards a smaller final batch, which may be appropriate for shape-sensitive training but changes how many examples contribute. Multiple workers can overlap loading with computation, yet each worker must receive appropriate random seeds and must not duplicate an iterable stream.

Pinned host memory and asynchronous device copies can improve GPU pipelines, but only when the complete transfer path supports them. More workers are not always faster: small in-memory datasets may spend more time on process coordination than data preparation. The official [Datasets & DataLoaders tutorial](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html) provides the core API behavior.

**Application.** The data boundary is where tokenization, augmentation, padding, label conversion, and sample metadata become tensors. Testing the Dataset and collate function independently prevents model code from hiding malformed examples or inconsistent shapes.

**Comparison summary.** Dataset defines a sample, Sampler defines order, collate defines batch structure, and DataLoader orchestrates iteration. A mini-batch is both a gradient estimator and a hardware work unit.

### **Devices, Data Types, and Numerical Precision** {#devices-data-types-numerical-precision}

A tensor operation can execute only when its operands satisfy the kernel's device and dtype requirements. PyTorch does not silently move a CPU tensor to a GPU for arithmetic because an implicit transfer would be expensive and difficult to reason about. Model state and input batches must be moved deliberately.

Common device choices include:

- `cpu` for portability, preprocessing, small models, and control-heavy work;
- `cuda` for NVIDIA accelerators;
- `mps` for supported Apple Silicon acceleration;
- other backends such as XPU or specialized accelerators when available.

Device placement is not only about compute. A training step may be limited by host-to-device transfer, accelerator memory, kernel-launch overhead, or synchronization. Calling `.item()`, printing a GPU tensor, or moving data back to the CPU inside a tight loop can force synchronization and reduce throughput.

The **dtype** controls numerical range, precision, storage, and eligible kernels:

| Dtype | Bytes per element | Typical role | Main caution |
|---|---:|---|---|
| `float64` | 8 | scientific checks, high-precision diagnostics | slower and larger on many accelerators |
| `float32` | 4 | standard training and stable accumulation | more memory than reduced precision |
| `bfloat16` | 2 | mixed-precision training with float32-like exponent range | fewer mantissa bits |
| `float16` | 2 | accelerated mixed precision and inference | narrower exponent range, overflow/underflow risk |
| `int64` | 8 | class labels and indices expected by many APIs | not differentiable |
| `bool` | 1 | masks and logical selection | arithmetic meaning must be explicit |

A dense tensor's payload memory is approximately:

$$
\text{bytes}
=
\operatorname{numel}(X)
\times
\operatorname{element\_size}(X).
$$

This excludes allocator overhead and any gradients, optimizer states, saved activations, or temporary workspaces. A model parameter used with Adam may generate several additional tensors beyond the parameter itself.

Reduced precision accelerates many workloads and saves memory, but not every operation should run at the same dtype. Mixed-precision systems typically use autocasting, higher-precision accumulation, and gradient scaling where necessary. Chapter 19 treats that training system in depth.

Numerical stability depends on algorithm form as well as dtype. A naive softmax:

$$
p_i=\frac{e^{z_i}}{\sum_j e^{z_j}}
$$

can overflow when logits are large. Subtracting the maximum is mathematically equivalent and numerically safer:

$$
p_i
=
\frac{e^{z_i-m}}{\sum_j e^{z_j-m}},
\qquad
m=\max_j z_j.
$$

Production `torch.softmax` uses a stable implementation; manually expanding familiar formulas can lose that protection.

<details>
<summary><strong>PyTorch: choose a device, estimate tensor memory, and inspect stability</strong></summary>

~~~python
import torch
from torch import nn


def available_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


device = available_device()
model = nn.Linear(768, 10).to(device)
features = torch.randn(32, 128, 768, device=device)
logits = model(features)
assert logits.shape == (32, 128, 10)

payload_mebibytes = features.numel() * features.element_size() / 2**20
print("device:", device)
print("activation payload MiB:", round(payload_mebibytes, 2))

# A direct exponential overflows even though the probability is well defined.
large_logits = torch.tensor([1000.0, 1001.0])
naive = torch.exp(large_logits) / torch.exp(large_logits).sum()
stable = torch.softmax(large_logits, dim=0)

print("naive softmax:", naive)
print("stable softmax:", stable)
assert torch.isnan(naive).any()
assert torch.isfinite(stable).all()
~~~

</details>

The code moves the model and features to the same selected device. Targets must follow the loss contract: for `CrossEntropyLoss`, class-index targets are normally `int64`, while model logits are floating point. Casting every tensor to one dtype is not a valid device strategy.

**Comparison summary.** Device determines where computation occurs; dtype determines representation range, precision, and memory; algorithm form determines how rounding and exponent limits are encountered. Performance and stability require all three to be designed together.

### **Randomness, Checkpoints, and Reproducibility** {#randomness-checkpoints-reproducibility}

Deep learning experiments contain multiple kinds of state. Model parameters are only one part:

- Python, NumPy, and PyTorch random-number generator states;
- data split identities, sample order, and augmentation randomness;
- optimizer momentum and variance estimates;
- scheduler, gradient-scaler, and early-stopping state;
- training step, epoch, configuration, code version, and environment;
- model parameters and persistent buffers.

Setting seeds makes a run more controlled, but it does not promise bitwise equality across PyTorch releases, platforms, devices, or kernels. Some accelerator algorithms are nondeterministic, and deterministic alternatives can be slower. A reproducibility claim must therefore name the environment and the level of equivalence: exact replay, statistically consistent results, or reproduction of a scientific conclusion. The official [PyTorch reproducibility notes](https://docs.pytorch.org/docs/stable/notes/randomness.html) make this limitation explicit.

A **checkpoint** is a serialization of state needed to continue or inspect a run. Saving only `model.state_dict()` is sufficient for inference when the architecture and preprocessing are known. Exact training resumption generally also needs optimizer state, progress counters, random states, and the data-order state.

<details>
<summary><strong>PyTorch: create and restore a self-contained training checkpoint</strong></summary>

~~~python
import io
import random

import numpy as np
import torch
from torch import nn

SEED = 23
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

model = nn.Sequential(nn.Linear(3, 5), nn.Tanh(), nn.Linear(5, 1))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)
features = torch.randn(6, 3)
targets = torch.randn(6, 1)

# Perform one update so the optimizer also contains nontrivial state.
loss = nn.functional.mse_loss(model(features), targets)
optimizer.zero_grad()
loss.backward()
optimizer.step()
reference_prediction = model(features).detach().clone()

data_generator = torch.Generator().manual_seed(99)
checkpoint = {
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict(),
    "epoch": 1,
    "config": {"input_width": 3, "hidden_width": 5, "learning_rate": 1e-2},
    "python_rng_state": random.getstate(),
    "numpy_rng_state": np.random.get_state(),
    "torch_rng_state": torch.get_rng_state(),
    "data_generator_state": data_generator.get_state(),
}

# BytesIO keeps this example self-contained; a real run writes an atomic file.
buffer = io.BytesIO()
torch.save(checkpoint, buffer)
buffer.seek(0)

# weights_only=False is appropriate only for a trusted full-state checkpoint.
loaded = torch.load(buffer, map_location="cpu", weights_only=False)
restored_model = nn.Sequential(nn.Linear(3, 5), nn.Tanh(), nn.Linear(5, 1))
restored_optimizer = torch.optim.AdamW(restored_model.parameters(), lr=1e-2)
restored_model.load_state_dict(loaded["model_state"])
restored_optimizer.load_state_dict(loaded["optimizer_state"])

random.setstate(loaded["python_rng_state"])
np.random.set_state(loaded["numpy_rng_state"])
torch.set_rng_state(loaded["torch_rng_state"])
data_generator.set_state(loaded["data_generator_state"])

restored_prediction = restored_model(features).detach()
assert torch.allclose(reference_prediction, restored_prediction)
print("restored epoch:", loaded["epoch"])
print("predictions restored:", True)
~~~

</details>

Full checkpoints use Python serialization and must be loaded only from trusted sources. For distributing model weights, prefer a state dictionary and the safest available loading mode. A checkpoint also does not preserve external dependencies automatically: data files, vocabulary versions, preprocessing code, and architecture definitions must be versioned separately.

Operationally, checkpoint writing should be atomic: write a temporary file, flush it, and rename it only after success. Keep a latest resumable checkpoint separately from the best validation checkpoint, because they answer different needs. A corrupted or partially written "best" file should not be the only recovery path.

**Application.** Reproducibility supports debugging as much as publication. When a loss spike appears at step 18,400, a complete checkpoint and configuration can distinguish a data-dependent failure from a one-off hardware or numerical event.

**Comparison summary.** A seed controls an initial random stream, deterministic settings constrain algorithm choices, a checkpoint captures execution state, and an experiment record explains the environment and decisions. None substitutes for the others.

### **Chapter Comparison and Summary** {#chapter-comparison-summary}

Tensor programming becomes reliable when values are never separated from their contracts. Before an operation, identify every axis; after it, predict the output shape; for training, verify graph connectivity; for systems work, account for dtype, device, data order, and persistent state.

| Concept | Primary question | Frequent failure |
|---|---|---|
| Rank and shape | How many axes exist and how long are they? | confusing tensor rank with matrix rank |
| Axis semantics | What does movement along each axis change? | a legal permutation with the wrong meaning |
| Stride and layout | How do logical indices map to storage? | assuming every view is contiguous |
| Indexing and reshape | Which values remain and how are they grouped? | removing or merging the wrong semantic axis |
| Broadcasting | Which singleton axes are virtually expanded? | producing a valid but unintended larger tensor |
| Matrix contraction | Which axes are preserved and summed? | contracting feature, token, or head axes incorrectly |
| Computation graph | Which executed operations connect loss to parameters? | detaching or overwriting required values |
| Autograd | Where should reverse-mode sensitivities accumulate? | stale accumulated gradients or missing tracking |
| Module state | What must optimize, move, and serialize together? | unregistered tensors or direct `forward` calls |
| Data pipeline | How do samples become a valid batch? | padding without masks or duplicated worker streams |
| Device and dtype | Where and with what representation does a kernel run? | device mismatch, overflow, or inappropriate casting |
| Reproducibility | What state is required to explain or resume the run? | saving weights while losing optimizer and data order |

The main conclusions are:

1. A tensor is numerical data plus shape, dtype, device, layout, and optional autograd metadata.
2. Shape compatibility is necessary but not sufficient; axis semantics determine whether an operation is meaningful.
3. Views can share storage with different strides, while reshaping may require a copy.
4. Broadcasting aligns trailing dimensions and can create silent shape expansions.
5. Matrix multiplication and `einsum` are best understood as preserving some axes and contracting others.
6. PyTorch builds a dynamic computation graph from operations that actually execute.
7. Reverse-mode autograd composes local vector-Jacobian products and accumulates gradients in tracked leaves.
8. Modules register parameters, buffers, and submodules so state can be optimized, moved, and serialized coherently.
9. Dataset, sampling, collation, and device transfer form part of the model's behavioral contract.
10. Seeds, deterministic settings, checkpoints, and environment records solve different parts of reproducibility.

The next chapter uses these contracts to construct neural building blocks: linear transformations, activations, embeddings, residual paths, gates, normalization, and reusable modules.
